# Train the LedgerLens tabular GAN on a GPU

Trains the WGAN-GP in `ml/synthetic/gan/` on the seed invoice rows (CORD receipts +
vendor-master POs), saves checkpoints and `run.json` to Google Drive, then samples rows and
prints the quality table (`ml/synthetic/gan/evaluate.py`).

**Runtime → Change runtime type → T4 GPU.** On CPU the same commands work, just slower
(5,000 steps ≈ 8 minutes on a laptop CPU for ~1.5k seed rows).

Only `run.json` is meant to be committed; checkpoints (`*.pt`) stay on Drive.

In [ ]:
from google.colab import drive

drive.mount("/content/drive")
OUT = "/content/drive/MyDrive/ledgerlens/gan"

In [ ]:
!git clone --depth 1 https://github.com/Sam15b/LedgerLens.git /content/LedgerLens
%cd /content/LedgerLens
# Colab already ships a CUDA build of torch; install the package and the parquet reader.
!pip install -q -e . pyarrow
import torch

print(torch.__version__, "CUDA:", torch.cuda.is_available())

## Seed rows

Downloads CORD (~2.3 GB) and builds `data/synthetic/seed/seed_invoices.csv`. Skip the
download to train on the 500 vendor-master PO rows only.

In [ ]:
!python scripts/download_datasets.py cord
!python -m ml.synthetic.seed --out data/synthetic/seed

## Train

Adam (lr 2e-4, betas 0.5/0.9), 5 critic steps per generator step. Add
`--condition vendor_category` for the conditional GAN.

In [ ]:
STEPS = 5000
SEED = 0
!python ml/synthetic/gan/train.py --steps {STEPS} --seed {SEED} --device cuda --runs {OUT}/runs

In [ ]:
import json
from pathlib import Path

run_dir = max(Path(OUT, "runs").iterdir(), key=lambda p: p.stat().st_mtime)
run = json.loads((run_dir / "run.json").read_text())
print(run_dir, run["seconds"], "s")
for point in run["history"][-5:]:
    print(point)

## Sample and measure

Samples rows with the business-rule post-processing (`total = subtotal × (1 + tax_rate)`)
and compares them with the seed rows: per-column KS and Wasserstein distance, and the
difference between the two correlation matrices.

In [ ]:
!python -m ml.synthetic.gan.sample {run_dir}/final.pt -n 5000 --out {OUT}/samples.csv
!python -m ml.synthetic.gan.evaluate {run_dir}/final.pt -n 5000 --report {OUT}/gan.md
print(Path(OUT, "gan.md").read_text())